# Diabetic Retinopathy Classification: End-to-End ML Experiment

This notebook serves as the primary experiment pipeline for the APTOS 2019 Blindness Detection project. It trains and evaluates four deep learning architectures under standardized experimental conditions:

1. **ResNet50** (Transfer learning baseline)
2. **EfficientNet-B0** (Lightweight compound-scaled CNN)
3. **Dual-Backbone Fusion** (EfficientNet-B0 + DenseNet121 feature fusion)
4. **CNN + Transformer Hybrid** (ResNet34 feature extractor + Transformer Encoder)

### Experiment Workflow
* **Sections 1-3:** Environment setup, reproducibility seed, and DataLoader initialization
* **Section 4:** Sequential Model Construction Gate (validates shapes & parameters, frees memory)
* **Sections 5-6:** Training configuration & model experiment registry
* **Sections 7-8:** Sequential model training, checkpointing, and per-model training curve generation
* **Section 9:** Held-out test set evaluation (strictly post-training) & confusion matrix visualization
* **Sections 10-12:** Model comparison table, comparative visualizations, experiment metadata, and final summary


## 1. Environment & GPU Configuration

Mounts Google Drive, configures `PROJECT_ROOT`, inspects CUDA acceleration, and selects the active execution device.


In [ ]:
import os
import sys

# ---- 1. Mount Google Drive ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Running in local environment; skipping Google Drive mount.")
except Exception as e:
    print(f"Drive mount notice: {e}")

# ---- 2. Project Root Configuration ----
PROJECT_ROOT = "/content/drive/MyDrive/DR_project"

# Auto-detect path or search in MyDrive if shortcut has alternative casing/naming
if not os.path.isdir(PROJECT_ROOT):
    candidates = [
        "/content/drive/MyDrive/DR_project",
        "/content/drive/MyDrive/DR_Project",
        "/content/drive/Shareddrives/DR_project",
    ]
    for c in candidates:
        if os.path.isdir(c):
            PROJECT_ROOT = c
            break

# If still not found, search in /content/drive/MyDrive
if not os.path.isdir(PROJECT_ROOT) and os.path.isdir("/content/drive/MyDrive"):
    for item in os.listdir("/content/drive/MyDrive"):
        if "dr_project" in item.lower():
            PROJECT_ROOT = os.path.join("/content/drive/MyDrive", item)
            break

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"PROJECT_ROOT: {PROJECT_ROOT} (exists: {os.path.isdir(PROJECT_ROOT)})")

if not os.path.isdir(PROJECT_ROOT):
    print("WARNING: Project directory was not found!")
    if os.path.isdir("/content/drive/MyDrive"):
        print("Available items in /content/drive/MyDrive:")
        for it in sorted(os.listdir("/content/drive/MyDrive")):
            print(f"  - {it}")
    else:
        print("/content/drive/MyDrive does not exist. Ensure Google Drive is mounted.")

# ---- 3. Hardware & PyTorch Inspection ----
import torch
import torchvision

cuda_available = torch.cuda.is_available()
cuda_count = torch.cuda.device_count() if cuda_available else 0
gpu_name = torch.cuda.get_device_name(0) if cuda_available else "None"

print("=" * 65)
print("              HARDWARE & ENVIRONMENT CONFIGURATION             ")
print("=" * 65)
print(f"  Python Version:       {sys.version.split()[0]}")
print(f"  PyTorch Version:      {torch.__version__}")
print(f"  Torchvision Version:  {torchvision.__version__}")
print(f"  CUDA Available:       {cuda_available}")
print(f"  CUDA Device Count:    {cuda_count}")
print(f"  GPU Device Name:      {gpu_name}")
print("=" * 65)

if not cuda_available:
    print("WARNING: CUDA is not available. Experiments will run on CPU, which is substantially slower.")
    device = torch.device("cpu")
else:
    device = torch.device("cuda:0")
    print(f"Active Device: {device} ({gpu_name})")


Mounted at /content/drive
Google Drive mounted successfully.
PROJECT_ROOT: /content/drive/MyDrive/DR_project (exists: True)
              HARDWARE & ENVIRONMENT CONFIGURATION             
  Python Version:       3.13.15
  PyTorch Version:      2.11.0+cpu
  Torchvision Version:  0.26.0+cpu
  CUDA Available:       False
  CUDA Device Count:    0
  GPU Device Name:      None


## 2. Imports & Reproducibility

Sets fixed random seeds across Python, NumPy, and PyTorch (`SEED = 42`) and configures deterministic cuDNN settings.


In [ ]:
import os
import sys

# ============================================================
# BULLETPROOF PROJECT PATH & IMPORT SETUP
# ============================================================

# 1. Establish PROJECT_ROOT
PROJECT_ROOT = "/content/drive/MyDrive/DR_project"

# Auto-detect if the path has different casing or location
if not os.path.isdir(PROJECT_ROOT):
    search_dirs = ["/content/drive/MyDrive", "/content/drive/Shareddrives"]
    for search_dir in search_dirs:
        if os.path.isdir(search_dir):
            for item in os.listdir(search_dir):
                if "dr_project" in item.lower():
                    PROJECT_ROOT = os.path.join(search_dir, item)
                    break
        if os.path.isdir(PROJECT_ROOT):
            break

assert os.path.isdir(PROJECT_ROOT), (
    f"PROJECT_ROOT not found: {PROJECT_ROOT}. "
    f"Please verify the DR_project folder shortcut is in My Drive."
)

# 2. Add to sys.path (at position 0 for priority)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 3. Clear any previously cached (possibly failed) src module imports
for mod_name in list(sys.modules.keys()):
    if mod_name == "src" or mod_name.startswith("src."):
        del sys.modules[mod_name]

# 4. Force Python to re-scan directories (critical for Google Drive FUSE)
import importlib
importlib.invalidate_caches()

# 5. Ensure all __init__.py package markers exist
init_dirs = [
    os.path.join(PROJECT_ROOT, "src"),
    os.path.join(PROJECT_ROOT, "src", "data"),
    os.path.join(PROJECT_ROOT, "src", "models"),
    os.path.join(PROJECT_ROOT, "src", "training"),
    os.path.join(PROJECT_ROOT, "src", "evaluation"),
]
for init_dir in init_dirs:
    init_file = os.path.join(init_dir, "__init__.py")
    if not os.path.isfile(init_file):
        with open(init_file, "w") as f:
            f.write(f"# Auto-created package marker for {os.path.basename(init_dir)}\n")
        print(f"  Created missing: {init_file}")

# 6. Verify key source files exist before importing
required_files = [
    "src/data/dataloader.py",
    "src/models/resnet50.py",
    "src/models/efficientnet.py",
    "src/models/fusion.py",
    "src/models/cnn_transformer.py",
    "src/training/trainer.py",
    "src/evaluation/evaluator.py",
    "src/evaluation/metrics.py",
]
for rf in required_files:
    full_path = os.path.join(PROJECT_ROOT, rf)
    assert os.path.isfile(full_path), f"Required source file missing: {full_path}"

print(f"PROJECT_ROOT verified: {PROJECT_ROOT}")
print(f"sys.path[0]: {sys.path[0]}")
print(f"All {len(required_files)} source files found.")

# ============================================================
# STANDARD LIBRARY & THIRD-PARTY IMPORTS
# ============================================================
import copy
import gc
import json
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ============================================================
# PROJECT MODULE IMPORTS
# ============================================================
from src.data.dataloader import get_dataloaders
from src.models.resnet50 import build_resnet50
from src.models.efficientnet import build_efficientnet_b0
from src.models.fusion import build_fusion_model
from src.models.cnn_transformer import build_cnn_transformer
from src.training.trainer import Trainer, set_seed
from src.evaluation.evaluator import evaluate_model, count_parameters
from src.evaluation.metrics import DR_CLASS_NAMES

print("All project modules imported successfully.")

# ---- Set Global Random Seed ----
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

print("=" * 65)
print(f"Global Random Seed set to: {RANDOM_SEED}")
print("CuDNN Deterministic: True | CuDNN Benchmark: False")
print("=" * 65)

# ---- Create Project Output Directories ----
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
PLOTS_DIR = os.path.join(PROJECT_ROOT, "results", "plots")
METRICS_DIR = os.path.join(PROJECT_ROOT, "results", "metrics")

for directory in [CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR]:
    os.makedirs(directory, exist_ok=True)
    print(f"Verified directory: {directory}")


PROJECT_ROOT verified: /content/drive/MyDrive/DR_project
sys.path[0]: /content/drive/MyDrive/DR_project
All 8 source files found.
All project modules imported successfully.
Global Random Seed set to: 42
CuDNN Deterministic: True | CuDNN Benchmark: False
Verified directory: /content/drive/MyDrive/DR_project/checkpoints
Verified directory: /content/drive/MyDrive/DR_project/results/plots
Verified directory: /content/drive/MyDrive/DR_project/results/metrics


## 3. DataLoaders & Class Weights Calculation

Initializes training (2,563), validation (549), and test (550) DataLoaders via `get_dataloaders()`. Class weights are computed strictly from the training set using inverse class frequency to handle class imbalance.


In [ ]:
SPLITS_DIR = os.path.join(PROJECT_ROOT, "src", "data", "splits")
RAW_IMG_DIR = os.path.join(PROJECT_ROOT, "src", "data", "raw", "train_images")

# Verify data availability
for split_file in ["train.csv", "val.csv", "test.csv"]:
    split_path = os.path.join(SPLITS_DIR, split_file)
    assert os.path.isfile(split_path), f"Missing split file: {split_path}"
assert os.path.isdir(RAW_IMG_DIR), f"Missing raw images directory: {RAW_IMG_DIR}"

BATCH_SIZE = 32
num_workers = 2 if os.name != 'nt' else 0

# Create DataLoaders
dataloaders = get_dataloaders(
    splits_dir=SPLITS_DIR,
    img_dir=RAW_IMG_DIR,
    batch_size=BATCH_SIZE,
    num_workers=num_workers,
    img_size=224,
    pin_memory=torch.cuda.is_available()
)

train_loader = dataloaders["train"]
val_loader = dataloaders["val"]
test_loader = dataloaders["test"]

train_size = len(train_loader.dataset)
val_size = len(val_loader.dataset)
test_size = len(test_loader.dataset)

# Calculate Class Weights strictly on training data
train_targets = train_loader.dataset.df["diagnosis"].values
class_counts = np.bincount(train_targets, minlength=5)
total_samples = len(train_targets)
num_classes = 5

class_weights_np = total_samples / (num_classes * class_counts)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32)

print("=" * 65)
print("                   DATASET SIZES & CLASS WEIGHTS               ")
print("=" * 65)
print(f"  Train Samples:      {train_size:>5}")
print(f"  Validation Samples: {val_size:>5}")
print(f"  Test Samples:       {test_size:>5}")
print(f"  Batch Size:         {BATCH_SIZE:>5}")
print("-" * 65)
print("  Class Distribution & Inverse Frequency Weights (Train only):")
for cid in range(num_classes):
    print(f"    Class {cid} ({DR_CLASS_NAMES[cid]:<18}): Count = {class_counts[cid]:>4} | Weight = {class_weights_np[cid]:.4f}")
print("=" * 65)


                   DATASET SIZES & CLASS WEIGHTS               
  Train Samples:       2563
  Validation Samples:   549
  Test Samples:         550
  Batch Size:            32
-----------------------------------------------------------------
  Class Distribution & Inverse Frequency Weights (Train only):
    Class 0 (0 - No DR         ): Count = 1263 | Weight = 0.4059
    Class 1 (1 - Mild          ): Count =  259 | Weight = 1.9792
    Class 2 (2 - Moderate      ): Count =  699 | Weight = 0.7333
    Class 3 (3 - Severe        ): Count =  135 | Weight = 3.7970
    Class 4 (4 - Proliferative DR): Count =  207 | Weight = 2.4763


## 4. Model Construction Tests (Gate Check)

Tests all four models sequentially to verify architecture construction, forward pass execution, output tensor dimensions `[2, 5]`, absence of `NaN`/`Inf` values, and parameter counts.

Each model is evaluated in isolation and immediately freed from GPU memory. The gate halts execution before training if any model fails.


In [ ]:
model_test_specs = [
    (
        "ResNet50",
        lambda: build_resnet50(num_classes=5, pretrained=True, freeze_backbone=True)
    ),
    (
        "EfficientNet-B0",
        lambda: build_efficientnet_b0(num_classes=5, pretrained=True, freeze_backbone=True)
    ),
    (
        "Dual-Backbone Fusion",
        lambda: build_fusion_model(num_classes=5, pretrained=True, freeze_backbones=True)
    ),
    (
        "CNN + Transformer",
        lambda: build_cnn_transformer(num_classes=5, pretrained=True, freeze_cnn_backbone=True)
    ),
]

test_results = []
all_passed = True
failure_reasons = []

print("Running sequential model construction tests...")

for name, builder_fn in model_test_specs:
    model = None
    dummy_input = None
    output = None
    try:
        # 1. Construct model
        model = builder_fn()
        model = model.to(device)
        model.eval()

        # 2. Count parameters
        total_p, trainable_p = count_parameters(model)

        # 3. Dummy forward pass
        dummy_input = torch.randn(2, 3, 224, 224, device=device)
        with torch.no_grad():
            output = model(dummy_input)

        # 4. Verify output shape [2, 5]
        output_shape = list(output.shape)
        shape_valid = (output_shape == [2, 5])

        # 5. Verify no NaN / Inf
        has_nan_inf = bool(torch.isnan(output).any() or torch.isinf(output).any())

        if shape_valid and not has_nan_inf:
            status = "PASS"
        else:
            status = "FAIL"
            all_passed = False
            reasons = []
            if not shape_valid:
                reasons.append(f"Bad shape {output_shape}")
            if has_nan_inf:
                reasons.append("Contains NaN/Inf")
            failure_reasons.append(f"{name}: {', '.join(reasons)}")

        test_results.append({
            "Model": name,
            "Output Shape": str(output_shape),
            "Total Params": f"{total_p:,}",
            "Trainable Params": f"{trainable_p:,}",
            "Test Status": status
        })

    except Exception as e:
        all_passed = False
        failure_reasons.append(f"{name}: {str(e)}")
        test_results.append({
            "Model": name,
            "Output Shape": "N/A",
            "Total Params": "N/A",
            "Trainable Params": "N/A",
            "Test Status": f"FAIL ({type(e).__name__})"
        })

    finally:
        # Free memory before testing the next model
        if model is not None:
            model.to("cpu")
            del model
        if dummy_input is not None:
            del dummy_input
        if output is not None:
            del output
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Display summary table
df_gate = pd.DataFrame(test_results)
print("\n" + "=" * 80)
print("                      MODEL CONSTRUCTION GATE REPORT                    ")
print("=" * 80)
print(df_gate.to_string(index=False))
print("=" * 80)

if not all_passed:
    print("\nCRITICAL ERROR: One or more model construction tests failed!")
    for reason in failure_reasons:
        print(f"  - {reason}")
    raise RuntimeError(f"Model construction test gate failed: {failure_reasons}")
else:
    print("\nGATE CHECK PASSED: All 4 models verified. Safe to proceed to training.")


Running sequential model construction tests...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 154MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 94.8MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 127MB/s]
/content/drive/MyDrive/DR_project/src/models/cnn_transformer.py:113: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



                      MODEL CONSTRUCTION GATE REPORT                    
               Model Output Shape Total Params Trainable Params Test Status
            ResNet50       [2, 5]   23,518,277           10,245        PASS
     EfficientNet-B0       [2, 5]    4,013,953            6,405        PASS
Dual-Backbone Fusion       [2, 5]   12,145,153        1,183,749        PASS
   CNN + Transformer       [2, 5]    7,509,121        3,501,573        PASS

GATE CHECK PASSED: All 4 models verified. Safe to proceed to training.


## 5 & 6. Training Configuration & Model Specifications

Defines common training hyperparameters (15 epochs, early stopping patience 3, AdamW optimizer, learning rate 1e-3, weight decay 1e-4) and registers the 4 model configurations.


In [ ]:
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3
INITIAL_LR = 1e-3
WEIGHT_DECAY = 1e-4

EXPERIMENT_CONFIGS = [
    {
        "name": "resnet50",
        "display_name": "ResNet50",
        "builder": build_resnet50,
        "builder_kwargs": {
            "num_classes": 5,
            "pretrained": True,
            "freeze_backbone": True
        },
        "checkpoint_path": os.path.join(CHECKPOINT_DIR, "resnet50_best.pth"),
        "history_csv": os.path.join(METRICS_DIR, "resnet50_training_history.csv"),
        "loss_plot": os.path.join(PLOTS_DIR, "resnet50_loss.png"),
        "acc_plot": os.path.join(PLOTS_DIR, "resnet50_accuracy.png"),
        "cm_plot": os.path.join(PLOTS_DIR, "resnet50_confusion_matrix.png"),
    },
    {
        "name": "efficientnet_b0",
        "display_name": "EfficientNet-B0",
        "builder": build_efficientnet_b0,
        "builder_kwargs": {
            "num_classes": 5,
            "pretrained": True,
            "freeze_backbone": True
        },
        "checkpoint_path": os.path.join(CHECKPOINT_DIR, "efficientnet_b0_best.pth"),
        "history_csv": os.path.join(METRICS_DIR, "efficientnet_b0_training_history.csv"),
        "loss_plot": os.path.join(PLOTS_DIR, "efficientnet_b0_loss.png"),
        "acc_plot": os.path.join(PLOTS_DIR, "efficientnet_b0_accuracy.png"),
        "cm_plot": os.path.join(PLOTS_DIR, "efficientnet_b0_confusion_matrix.png"),
    },
    {
        "name": "efficientnet_densenet_fusion",
        "display_name": "Dual-Backbone Fusion",
        "builder": build_fusion_model,
        "builder_kwargs": {
            "num_classes": 5,
            "pretrained": True,
            "freeze_backbones": True
        },
        "checkpoint_path": os.path.join(CHECKPOINT_DIR, "fusion_best.pth"),
        "history_csv": os.path.join(METRICS_DIR, "fusion_training_history.csv"),
        "loss_plot": os.path.join(PLOTS_DIR, "fusion_loss.png"),
        "acc_plot": os.path.join(PLOTS_DIR, "fusion_accuracy.png"),
        "cm_plot": os.path.join(PLOTS_DIR, "fusion_confusion_matrix.png"),
    },
    {
        "name": "cnn_transformer",
        "display_name": "CNN + Transformer",
        "builder": build_cnn_transformer,
        "builder_kwargs": {
            "num_classes": 5,
            "pretrained": True,
            "freeze_cnn_backbone": True
        },
        "checkpoint_path": os.path.join(CHECKPOINT_DIR, "cnn_transformer_best.pth"),
        "history_csv": os.path.join(METRICS_DIR, "cnn_transformer_training_history.csv"),
        "loss_plot": os.path.join(PLOTS_DIR, "cnn_transformer_loss.png"),
        "acc_plot": os.path.join(PLOTS_DIR, "cnn_transformer_accuracy.png"),
        "cm_plot": os.path.join(PLOTS_DIR, "cnn_transformer_confusion_matrix.png"),
    }
]

print("Registered Experiments:")
for i, cfg in enumerate(EXPERIMENT_CONFIGS, 1):
    print(f"  {i}. {cfg['display_name']:<22} -> Checkpoint: {os.path.basename(cfg['checkpoint_path'])}")


Registered Experiments:
  1. ResNet50               -> Checkpoint: resnet50_best.pth
  2. EfficientNet-B0        -> Checkpoint: efficientnet_b0_best.pth
  3. Dual-Backbone Fusion   -> Checkpoint: fusion_best.pth
  4. CNN + Transformer      -> Checkpoint: cnn_transformer_best.pth


## 7 & 8. Training Experiments & Training Curves

Trains the four models sequentially using the existing `Trainer` class. To prevent GPU out-of-memory errors:
1. Each model is trained in isolation.
2. Best checkpoint is saved and history recorded.
3. Training/validation loss and accuracy curves are plotted and saved.
4. Model, optimizer, and trainer references are deleted, followed by `gc.collect()` and `torch.cuda.empty_cache()` before the next model is started.


In [ ]:
def plot_and_save_training_curves(history: dict, loss_path: str, acc_path: str, model_name: str):
    """Generates and saves separate training/validation loss and accuracy plots."""
    epochs = range(1, len(history["train_loss"]) + 1)

    # 1. Loss Plot
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], 'o-', color='#1f77b4', label='Train Loss', linewidth=2)
    plt.plot(epochs, history["val_loss"], 's-', color='#ff7f0e', label='Validation Loss', linewidth=2)
    plt.title(f"{model_name} - Training & Validation Loss", fontsize=14, fontweight='bold')
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Cross-Entropy Loss", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(loss_path, dpi=300)
    plt.close()

    # 2. Accuracy Plot
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_accuracy"], 'o-', color='#2ca02c', label='Train Accuracy', linewidth=2)
    plt.plot(epochs, history["val_accuracy"], 's-', color='#d62728', label='Validation Accuracy', linewidth=2)
    plt.title(f"{model_name} - Training & Validation Accuracy", fontsize=14, fontweight='bold')
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(acc_path, dpi=300)
    plt.close()

# Track experiment status
trained_models = {}
failed_models = {}

print("=" * 65)
print("               STARTING SEQUENTIAL MODEL TRAINING              ")
print("=" * 65)

for exp_idx, exp in enumerate(EXPERIMENT_CONFIGS, 1):
    name = exp["name"]
    display_name = exp["display_name"]
    target_ckpt = exp["checkpoint_path"]
    history_csv = exp["history_csv"]
    loss_plot = exp["loss_plot"]
    acc_plot = exp["acc_plot"]

    print(f"\n{'=' * 65}")
    print(f"  [{exp_idx}/{len(EXPERIMENT_CONFIGS)}] TRAINING: {display_name} ({name})")
    print(f"{'=' * 65}")

    # Check if this model was already completed in a previous run
    if os.path.exists(target_ckpt) and os.path.exists(history_csv):
        print(f"  Existing checkpoint and history found for {display_name}:")
        print(f"    Checkpoint: {target_ckpt}")
        print(f"    History:    {history_csv}")
        print(f"  Skipping training for {display_name} and loading recorded metrics.")
        df_prev = pd.read_csv(history_csv)
        best_val_loss = float(df_prev["val_loss"].min())
        best_epoch = int(df_prev.loc[df_prev["val_loss"] == best_val_loss, "epoch"].values[0])
        best_val_acc = float(df_prev.loc[df_prev["val_loss"] == best_val_loss, "val_accuracy"].values[0])
        trained_models[name] = {
            "config": exp,
            "best_val_loss": best_val_loss,
            "best_val_acc": best_val_acc,
            "best_epoch": best_epoch,
            "training_time": 0.0,
        }
        continue

    model = None
    optimizer = None
    scheduler = None
    trainer = None
    train_start_time = time.time()

    try:
        # 1. Construct model
        model = exp["builder"](**exp["builder_kwargs"])
        model = model.to(device)

        total_p, trainable_p = count_parameters(model)
        print(f"  Model:                {display_name}")
        print(f"  Total Parameters:     {total_p:,}")
        print(f"  Trainable Parameters: {trainable_p:,}")

        # 2. Setup optimizer & scheduler
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.1)

        # 3. Model-specific checkpoint directory
        model_ckpt_dir = os.path.join(CHECKPOINT_DIR, name)
        os.makedirs(model_ckpt_dir, exist_ok=True)

        # 4. Instantiate Trainer and execute fit()
        trainer = Trainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            class_weights=class_weights,
            device=device,
            num_epochs=MAX_EPOCHS,
            checkpoint_dir=model_ckpt_dir,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            seed=RANDOM_SEED
        )

        history = trainer.fit()
        train_elapsed = time.time() - train_start_time

        # 5. Copy best_model.pth to the required target checkpoint path
        saved_best_ckpt = os.path.join(model_ckpt_dir, "best_model.pth")
        if os.path.exists(saved_best_ckpt):
            shutil.copyfile(saved_best_ckpt, target_ckpt)
            print(f"  Checkpoint confirmed at: {target_ckpt}")
        else:
            print(f"  WARNING: Checkpoint {saved_best_ckpt} was not created.")

        # 6. Save training history to CSV
        df_history = pd.DataFrame(history)
        df_history.insert(0, "epoch", range(1, len(df_history) + 1))
        df_history.to_csv(history_csv, index=False)
        print(f"  Training history saved to: {history_csv}")

        # 7. Generate and save training plots
        plot_and_save_training_curves(history, loss_plot, acc_plot, display_name)
        print(f"  Plots saved: {os.path.basename(loss_plot)}, {os.path.basename(acc_plot)}")

        # 8. Report Best Validation Result
        best_val_loss = min(history["val_loss"])
        best_epoch = history["val_loss"].index(best_val_loss) + 1
        best_val_acc = history["val_accuracy"][best_epoch - 1]

        print("\n" + "-" * 65)
        print("BEST VALIDATION RESULT")
        print(f"Model:                    {display_name}")
        print(f"Best validation loss:     {best_val_loss:.4f}")
        print(f"Best validation accuracy: {best_val_acc:.4f}")
        print(f"Epoch:                    {best_epoch}")
        print(f"Training time:            {train_elapsed:.1f}s ({train_elapsed / 60:.1f} min)")
        print(f"Checkpoint:               {target_ckpt}")
        print("-" * 65)

        trained_models[name] = {
            "config": exp,
            "best_val_loss": best_val_loss,
            "best_val_acc": best_val_acc,
            "best_epoch": best_epoch,
            "training_time": train_elapsed,
        }

    except Exception as e:
        print(f"\nERROR during training of {display_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        failed_models[name] = str(e)

    finally:
        # Clean up GPU memory
        if model is not None:
            model.to("cpu")
            del model
        if optimizer is not None:
            del optimizer
        if scheduler is not None:
            del scheduler
        if trainer is not None:
            del trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n" + "=" * 65)
print(f"TRAINING PHASE COMPLETE: {len(trained_models)} succeeded, {len(failed_models)} failed.")
print("=" * 65)


               STARTING SEQUENTIAL MODEL TRAINING              

  [1/4] TRAINING: ResNet50 (resnet50)
  Existing checkpoint and history found for ResNet50:
    Checkpoint: /content/drive/MyDrive/DR_project/checkpoints/resnet50_best.pth
    History:    /content/drive/MyDrive/DR_project/results/metrics/resnet50_training_history.csv
  Skipping training for ResNet50 and loading recorded metrics.

  [2/4] TRAINING: EfficientNet-B0 (efficientnet_b0)
  Existing checkpoint and history found for EfficientNet-B0:
    Checkpoint: /content/drive/MyDrive/DR_project/checkpoints/efficientnet_b0_best.pth
    History:    /content/drive/MyDrive/DR_project/results/metrics/efficientnet_b0_training_history.csv
  Skipping training for EfficientNet-B0 and loading recorded metrics.

  [3/4] TRAINING: Dual-Backbone Fusion (efficientnet_densenet_fusion)
  Existing checkpoint and history found for Dual-Backbone Fusion:
    Checkpoint: /content/drive/MyDrive/DR_project/checkpoints/fusion_best.pth
    History:   

## 9. Final Test Evaluation (Held-Out Test Set Only)

**CRITICAL:** Only after all four models have completed training are they evaluated on the unseen `test_loader` (550 samples). Each model is reconstructed, its best checkpoint is restored, and comprehensive classification metrics are calculated via `evaluate_model()`.


In [ ]:
def plot_confusion_matrix(cm, class_names, save_path, model_name):
    """Plots and saves a formatted confusion matrix heatmap."""
    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True
    )
    plt.title(f"{model_name} - Confusion Matrix (Test Set)", fontsize=14, fontweight='bold')
    plt.xlabel("Predicted Class", fontsize=12)
    plt.ylabel("True Class", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

evaluation_results = {}
evaluated_models = {}

print("=" * 65)
print("             STARTING FINAL HELD-OUT TEST EVALUATION           ")
print("=" * 65)

for exp in EXPERIMENT_CONFIGS:
    name = exp["name"]
    display_name = exp["display_name"]
    ckpt_path = exp["checkpoint_path"]
    cm_plot_path = exp["cm_plot"]

    if name not in trained_models:
        print(f"Skipping evaluation for {display_name} (Training failed or was not completed).")
        continue

    print(f"\nEvaluating {display_name} on held-out Test Set ({test_size} samples)...")

    model = None
    try:
        # 1. Reconstruct model
        model = exp["builder"](**exp["builder_kwargs"])

        # 2. Load best checkpoint
        assert os.path.isfile(ckpt_path), f"Checkpoint missing: {ckpt_path}"
        checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(f"  Loaded checkpoint from: {ckpt_path}")

        # 3. Evaluate on test_loader
        eval_output = evaluate_model(
            model=model,
            dataloader=test_loader,
            device=device,
            num_classes=5,
            verbose=True
        )

        metrics = eval_output["metrics"]

        # 4. Save Confusion Matrix plot
        cm = np.array(metrics["confusion_matrix"])
        plot_confusion_matrix(cm, DR_CLASS_NAMES, cm_plot_path, display_name)
        print(f"  Confusion matrix plot saved: {cm_plot_path}")

        # 5. Store evaluation metrics
        evaluation_results[name] = {
            "Model": display_name,
            "Accuracy": metrics["accuracy"],
            "Macro Precision": metrics["precision_macro"],
            "Weighted Precision": metrics["precision_weighted"],
            "Macro Recall": metrics["recall_macro"],
            "Weighted Recall": metrics["recall_weighted"],
            "Macro F1": metrics["f1_macro"],
            "Weighted F1": metrics["f1_weighted"],
            "Macro Sensitivity": metrics["sensitivity_macro"],
            "Macro Specificity": metrics["specificity_macro"],
            "Macro ROC-AUC": metrics["roc_auc_macro"] if metrics["roc_auc_macro"] is not None else float("nan"),
            "Weighted ROC-AUC": metrics["roc_auc_weighted"] if metrics["roc_auc_weighted"] is not None else float("nan"),
            "Quadratic Weighted Kappa": metrics["quadratic_weighted_kappa"],
            "Total Parameters": eval_output["total_parameters"],
            "Trainable Parameters": eval_output["trainable_parameters"],
            "Inference Time": eval_output["inference_time_seconds"],
        }
        evaluated_models[name] = True

    except Exception as e:
        print(f"ERROR during evaluation of {display_name}: {str(e)}")
        import traceback
        traceback.print_exc()

    finally:
        if model is not None:
            model.to("cpu")
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n" + "=" * 65)
print(f"EVALUATION COMPLETE: {len(evaluated_models)} of {len(trained_models)} models evaluated.")
print("=" * 65)


             STARTING FINAL HELD-OUT TEST EVALUATION           

Evaluating ResNet50 on held-out Test Set (550 samples)...
  Loaded checkpoint from: /content/drive/MyDrive/DR_project/checkpoints/resnet50_best.pth
                  EVALUATION REPORT                    
  Samples Evaluated:            550
  Total Parameters:      23,518,277
  Trainable Parameters:      10,245
  Inference Time:            269.66s
  Throughput:                   2.0 samples/s

              CLASSIFICATION METRICS REPORT             
  Accuracy:                    0.6891
  Precision (macro):           0.5473
  Precision (weighted):        0.7742
  Recall (macro):              0.5919
  Recall (weighted):           0.6891
  F1 Score (macro):            0.5225
  F1 Score (weighted):         0.6876
  Sensitivity (macro):         0.5919
  Specificity (macro):         0.9277
  Quadratic Weighted Kappa:    0.8058
  ROC-AUC (macro):             0.9106
  ROC-AUC (weighted):          0.9421
--------------------------

/content/drive/MyDrive/DR_project/src/models/cnn_transformer.py:113: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(


  Loaded checkpoint from: /content/drive/MyDrive/DR_project/checkpoints/cnn_transformer_best.pth
                  EVALUATION REPORT                    
  Samples Evaluated:            550
  Total Parameters:       7,509,121
  Trainable Parameters:   3,501,573
  Inference Time:            117.05s
  Throughput:                   4.7 samples/s

              CLASSIFICATION METRICS REPORT             
  Accuracy:                    0.7218
  Precision (macro):           0.5523
  Precision (weighted):        0.7539
  Recall (macro):              0.6056
  Recall (weighted):           0.7218
  F1 Score (macro):            0.5207
  F1 Score (weighted):         0.7150
  Sensitivity (macro):         0.6056
  Specificity (macro):         0.9323
  Quadratic Weighted Kappa:    0.8298
  ROC-AUC (macro):             0.9070
  ROC-AUC (weighted):          0.9330
------------------------------------------------------------
  Per-Class Sensitivity & Specificity:
    0 - No DR                  Sens=0.9631

## 10. Final Model Comparison Table

Assembles all evaluation metrics into a pandas DataFrame, displays the full comparison, and saves the results to CSV and formatted plain-text formats.


In [ ]:
comparison_columns = [
    "Model",
    "Accuracy",
    "Macro Precision",
    "Weighted Precision",
    "Macro Recall",
    "Weighted Recall",
    "Macro F1",
    "Weighted F1",
    "Macro Sensitivity",
    "Macro Specificity",
    "Macro ROC-AUC",
    "Weighted ROC-AUC",
    "Quadratic Weighted Kappa",
    "Total Parameters",
    "Trainable Parameters",
    "Inference Time"
]

rows = [evaluation_results[m["name"]] for m in EXPERIMENT_CONFIGS if m["name"] in evaluation_results]
df_comparison = pd.DataFrame(rows, columns=comparison_columns)

# Save to CSV
comparison_csv_path = os.path.join(METRICS_DIR, "final_model_comparison.csv")
df_comparison.to_csv(comparison_csv_path, index=False)
print(f"Saved comparison CSV to: {comparison_csv_path}")

# Save readable text format
comparison_txt_path = os.path.join(METRICS_DIR, "final_model_comparison.txt")
with open(comparison_txt_path, "w") as f:
    f.write("=" * 100 + "\n")
    f.write("                   FINAL MODEL COMPARISON (APTOS 2019 TEST SET)                \n")
    f.write("=" * 100 + "\n")
    f.write(df_comparison.to_string(index=False) + "\n")
    f.write("=" * 100 + "\n")
print(f"Saved comparison text to: {comparison_txt_path}")

print("\n" + "=" * 100)
print("                                FINAL MODEL COMPARISON                              ")
print("=" * 100)
display_df = df_comparison.copy()
numeric_cols = [c for c in display_df.columns if c not in ["Model", "Total Parameters", "Trainable Parameters"]]
for col in numeric_cols:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if isinstance(x, (int, float)) and not np.isnan(x) else "N/A")
for col in ["Total Parameters", "Trainable Parameters"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x:,}")

print(display_df.to_string(index=False))
print("=" * 100)


Saved comparison CSV to: /content/drive/MyDrive/DR_project/results/metrics/final_model_comparison.csv
Saved comparison text to: /content/drive/MyDrive/DR_project/results/metrics/final_model_comparison.txt

                                FINAL MODEL COMPARISON                              
               Model Accuracy Macro Precision Weighted Precision Macro Recall Weighted Recall Macro F1 Weighted F1 Macro Sensitivity Macro Specificity Macro ROC-AUC Weighted ROC-AUC Quadratic Weighted Kappa Total Parameters Trainable Parameters Inference Time
            ResNet50   0.6891          0.5473             0.7742       0.5919          0.6891   0.5225      0.6876            0.5919            0.9277        0.9106           0.9421                   0.8058       23,518,277               10,245       269.6642
     EfficientNet-B0   0.7345          0.5697             0.7494       0.5776          0.7345   0.5659      0.7364            0.5776            0.9327        0.9057           0.9321        

## 11. Final Visual Comparisons

Generates comparative bar charts for Accuracy, Macro F1, and Quadratic Weighted Kappa across all evaluated models.


In [ ]:
if not df_comparison.empty:
    models = df_comparison["Model"].tolist()
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(models)]

    # 1. Accuracy Comparison Plot
    plt.figure(figsize=(9, 5))
    bars = plt.bar(models, df_comparison["Accuracy"], color=colors, width=0.5, edgecolor='black', linewidth=1)
    plt.title("Model Test Accuracy Comparison", fontsize=14, fontweight='bold')
    plt.ylabel("Accuracy", fontsize=12)
    plt.ylim(0, 1.05)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02, f"{height:.4f}",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    acc_cmp_path = os.path.join(PLOTS_DIR, "model_accuracy_comparison.png")
    plt.savefig(acc_cmp_path, dpi=300)
    plt.close()
    print(f"Saved: {acc_cmp_path}")

    # 2. Macro F1 Comparison Plot
    plt.figure(figsize=(9, 5))
    bars = plt.bar(models, df_comparison["Macro F1"], color=colors, width=0.5, edgecolor='black', linewidth=1)
    plt.title("Model Test Macro F1-Score Comparison", fontsize=14, fontweight='bold')
    plt.ylabel("Macro F1-Score", fontsize=12)
    plt.ylim(0, 1.05)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02, f"{height:.4f}",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    f1_cmp_path = os.path.join(PLOTS_DIR, "model_f1_comparison.png")
    plt.savefig(f1_cmp_path, dpi=300)
    plt.close()
    print(f"Saved: {f1_cmp_path}")

    # 3. Quadratic Weighted Kappa Comparison Plot
    plt.figure(figsize=(9, 5))
    bars = plt.bar(models, df_comparison["Quadratic Weighted Kappa"], color=colors, width=0.5, edgecolor='black', linewidth=1)
    plt.title("Model Test Quadratic Weighted Kappa Comparison", fontsize=14, fontweight='bold')
    plt.ylabel("Quadratic Weighted Kappa", fontsize=12)
    plt.ylim(min(0, df_comparison["Quadratic Weighted Kappa"].min() - 0.05), 1.05)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02, f"{height:.4f}",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    kappa_cmp_path = os.path.join(PLOTS_DIR, "model_kappa_comparison.png")
    plt.savefig(kappa_cmp_path, dpi=300)
    plt.close()
    print(f"Saved: {kappa_cmp_path}")
else:
    print("No evaluated models available to plot visual comparisons.")


Saved: /content/drive/MyDrive/DR_project/results/plots/model_accuracy_comparison.png
Saved: /content/drive/MyDrive/DR_project/results/plots/model_f1_comparison.png
Saved: /content/drive/MyDrive/DR_project/results/plots/model_kappa_comparison.png


## 12. Save Experiment Metadata & Complete Summary

Records experimental hyperparameters and configuration metadata, and prints the final execution summary.


In [ ]:
# Save Experiment Metadata
metadata_path = os.path.join(METRICS_DIR, "experiment_config.txt")

with open(metadata_path, "w") as f:
    f.write("============================================================\n")
    f.write("           EXPERIMENT CONFIGURATION & METADATA              \n")
    f.write("============================================================\n")
    f.write("Dataset Name:              APTOS 2019 Blindness Detection\n")
    f.write("Number of Classes:         5\n")
    f.write(f"Split Sizes:               Train={train_size}, Val={val_size}, Test={test_size} (70/15/15)\n")
    f.write(f"Random Seed:               {RANDOM_SEED}\n")
    f.write("Image Size:                224 x 224\n")
    f.write(f"Batch Size:                {BATCH_SIZE}\n")
    f.write("Optimizer:                 AdamW\n")
    f.write(f"Initial Learning Rate:     {INITIAL_LR}\n")
    f.write(f"Weight Decay:              {WEIGHT_DECAY}\n")
    f.write(f"Maximum Epochs:            {MAX_EPOCHS}\n")
    f.write(f"Early Stopping Patience:   {EARLY_STOPPING_PATIENCE}\n")
    f.write("Scheduler:                 ReduceLROnPlateau(mode='min', patience=2, factor=0.1)\n")
    f.write("Loss Function:             CrossEntropyLoss (Inverse Frequency Weighted)\n")
    f.write(f"GPU Name:                  {gpu_name}\n")
    f.write(f"PyTorch Version:           {torch.__version__}\n")
    f.write(f"Torchvision Version:       {torchvision.__version__}\n\n")
    f.write("Model Configurations:\n")
    for exp in EXPERIMENT_CONFIGS:
        f.write(f"  - {exp['display_name']} ({exp['name']}):\n")
        for k, v in exp['builder_kwargs'].items():
            f.write(f"      {k}: {v}\n")
        f.write(f"      Checkpoint: {exp['checkpoint_path']}\n")
    f.write("============================================================\n")

print(f"Saved experiment metadata to: {metadata_path}")

# Complete Experiment Summary
print("\n" + "=" * 60)
print("COMPLETE EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Models successfully trained:   {list(trained_models.keys()) if trained_models else 'None'}")
print(f"Models successfully evaluated: {list(evaluated_models.keys()) if evaluated_models else 'None'}")
print(f"Models failed:                 {list(failed_models.keys()) if failed_models else 'None'}")
print(f"Final comparison saved to:     {comparison_csv_path}")
print(f"Training plots saved to:       {PLOTS_DIR}")
print(f"Evaluation plots saved to:     {PLOTS_DIR}")
print(f"Checkpoints saved to:          {CHECKPOINT_DIR}")
print("=" * 60)


Saved experiment metadata to: /content/drive/MyDrive/DR_project/results/metrics/experiment_config.txt

COMPLETE EXPERIMENT SUMMARY
Models successfully trained:   ['resnet50', 'efficientnet_b0', 'efficientnet_densenet_fusion', 'cnn_transformer']
Models successfully evaluated: ['resnet50', 'efficientnet_b0', 'efficientnet_densenet_fusion', 'cnn_transformer']
Models failed:                 None
Final comparison saved to:     /content/drive/MyDrive/DR_project/results/metrics/final_model_comparison.csv
Training plots saved to:       /content/drive/MyDrive/DR_project/results/plots
Evaluation plots saved to:     /content/drive/MyDrive/DR_project/results/plots
Checkpoints saved to:          /content/drive/MyDrive/DR_project/checkpoints
